In [1]:
import pandas as pd

file_path = "inputs.xlsx"
df = pd.read_excel(file_path)
df

ImportError: Unable to import required dependencies:
numpy: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.
The following compiled module files exist, but seem incompatible
with with either python 'cpython-310' or the platform 'win32':

  * _multiarray_umath.cp314-win_amd64.lib
  * _multiarray_umath.cp314-win_amd64.pyd

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python 3.10 from "c:\Users\Nahid\FAIR_guidance_instruction\venv\Scripts\python.exe"
  * The NumPy version is: "2.4.3"

and make sure that they are the versions you expect.

Please carefully study the information and documentation linked above.
This is unlikely to be a NumPy issue but will be caused by a bad i

In [4]:

# Function to create prompt from one row
def build_prompt(row):
    prompt = f"""
You are a FAIR data expert helping write instructions for making data FAIR.

Use file format: {row['use_file_format']}.
Tool(s) that can help with the conversion or processing: {row['related_tools']}.

Follow dataset structure: {row['dataset_structure']}.
Here are more details: {row['dataset_structure_details']}.

Include metadata files: {row['include_metadata_files']}.
Use this tool to prepare metadata: {row['tool_to_prepare_metadata']}.

De-identification required: {row['deidentification_required']}.

The following repositories are suggested by your funder for this data modality:
{row['repositories_suggested_by_funder']}.

As per your funder policy, you should use one of the following data license(s):
{row['data_license_required_by_funder']}.

Write clear, practical FAIR instructions in a concise paragraph or bullet format.
"""
    return prompt.strip()

# Create prompt column
df["prompt"] = df.apply(build_prompt, axis=1)

# Show first prompt
print(df["prompt"].iloc[0])

You are a FAIR data expert helping write instructions for making data FAIR.

Use file format: MRI, fMRI, H-fMRS imaging data; demographic and clinical tabular data.
Tool(s) that can help with the conversion or processing: Custom Python code using statsmodels, NumPy, and pandas; LC Model 6.3 using LCMgui; SPM8 toolbox for MATLAB; GitHub lab website; README.md with instructions and parameter choices.

Follow dataset structure: NDA data dictionaries / NDA submission structure.
Here are more details: Research Subject and Pedigree (ndar_subject01); Demographics Short Form (demsf01); Ethnic Group Questionnaire (ethgrp01); Height and Weight (height_weight01); Hollingshead Socioeconomic Rating Scale (ses01); Pubertal Development Scale (pds01); Edinburgh Handedness Inventory (edinburgh_hand01); WASI-2 (wasi201); DSM Crosscutting for Youth (dsm5crossch01); RCADS-25 (rcads2501); Kiddie-SADS-Present and Lifetime Version (ksads_pl01); Children’s Yale-Brown Obsessive Compulsive Scale (cybocs01); Sch

In [5]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def ask_gpt(prompt_text):
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": "You are a FAIR data expert helping generate practical FAIR instructions."
            },
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0
    )
    return response.choices[0].message.content

# Apply to each row
df["gpt_answer"] = df["prompt"].apply(ask_gpt)

# Save output
df.to_excel("FAIR_instructions_with_prompts_and_answers.xlsx", index=False)

In [11]:
file_path = "FAIR_instructions_with_prompts_and_answers.xlsx"

df1 = pd.read_excel(file_path)

df1

,link,id,use_file_format,related_tools,dataset_structure,dataset_structure_details,include_metadata_files,tool_to_prepare_metadata,deidentification_required,repositories_suggested_by_funder,data_license_required_by_funder,prompt,gpt_answer
0,https://grants.nih.gov/sites/default/files/flm...,1.0,"MRI, fMRI, H-fMRS imaging data; demographic an...","Custom Python code using statsmodels, NumPy, a...",NDA data dictionaries / NDA submission structure,Research Subject and Pedigree (ndar_subject01)...,Subject level data; H fMRS and fMRI task relat...,NDA GUID tool; NDA validation tool,yes,NIMH Data Archive (NDA),Not specified; controlled access through the N...,You are a FAIR data expert helping write instr...,"**FAIR Data Instructions for MRI, fMRI, H-fMRS..."
1,https://grants.nih.gov/sites/default/files/flm...,2.0,FASTQ; SAM/BAM; BED; VCF; phenotypic and clini...,Illumina sequencing pipelines; Broad Institute...,"NDA data dictionaries structure for genomics, ...",Genomics Subject (genomics_subject02); Genomic...,study protocol; Data Submission Agreement; Dat...,NDA GUID tool; NDA Validation and Upload tool,yes,NIMH Data Archive (NDA); dbGaP registration,Not specified; controlled access via NDA Data ...,You are a FAIR data expert helping write instr...,**FAIR Data Preparation Instructions for Genom...
2,https://grants.nih.gov/sites/default/files/flm...,3.0,single-cell data matrices (cells × features); ...,custom analysis pipelines; GitHub repository w...,NeMO data standards structure for single-cell ...,cell × feature matrix structure; metadata tabl...,supplementary protocol document; protocols.io ...,NeMO submission pipelines; protocols.io workfl...,yes (for human single-cell data),Neuroscience Multi-omic Archive (NeMO),Not specified; controlled access for human dat...,You are a FAIR data expert helping write instr...,**FAIR Data Instructions for Single-Cell Multi...
3,https://grants.nih.gov/sites/default/files/flm...,4.0,MRI imaging data; clinical assessment data; de...,R; FMRIB Software Library (FSL); BrainVoyager;...,"NDA data dictionaries structure for genomics, ...",ndar_subject01; cde_phq901; bdi01; image03; im...,Data Submission Agreement; Data Expected list;...,NDA GUID tool; NDA validation tool,yes,NIMH Data Archive (NDA),Data Use Certification (DUC); controlled acces...,You are a FAIR data expert helping write instr...,"**FAIR Data Instructions for MRI Imaging, Clin..."
4,https://bids.neuroimaging.io/index.html,NaN,NIfTI,dcm2niix,BIDS,bids-specification.readthedocs.io/en/stable,dataset_description.json participants.tsv part...,BIDS Starter Kit; HeuDiConv; ezBIDS; PyBIDS; B...,yes,OpenNeuro,CC-BY-4.0,You are a FAIR data expert helping write instr...,"**FAIR Data Instructions for MRI Data (NIfTI, ..."


In [12]:
for i, row in df1.iterrows():
    print(f"\n--- Row {i} ---")
    print("PROMPT:\n", row["prompt"])
    print("\nANSWER:\n", row["gpt_answer"])
    print("\n" + "="*60)


--- Row 0 ---
PROMPT:
 You are a FAIR data expert helping write instructions for making data FAIR.

Use file format: MRI, fMRI, H-fMRS imaging data; demographic and clinical tabular data.
Tool(s) that can help with the conversion or processing: Custom Python code using statsmodels, NumPy, and pandas; LC Model 6.3 using LCMgui; SPM8 toolbox for MATLAB; GitHub lab website; README.md with instructions and parameter choices.

Follow dataset structure: NDA data dictionaries / NDA submission structure.
Here are more details: Research Subject and Pedigree (ndar_subject01); Demographics Short Form (demsf01); Ethnic Group Questionnaire (ethgrp01); Height and Weight (height_weight01); Hollingshead Socioeconomic Rating Scale (ses01); Pubertal Development Scale (pds01); Edinburgh Handedness Inventory (edinburgh_hand01); WASI-2 (wasi201); DSM Crosscutting for Youth (dsm5crossch01); RCADS-25 (rcads2501); Kiddie-SADS-Present and Lifetime Version (ksads_pl01); Children’s Yale-Brown Obsessive Compulsi